# Telemetry explorer

Barebones: pick a session, pick channels, plot. Everything is a plain variable
you edit and re-run — no widgets to install.

**There is no oil temperature channel in this data.** The OBD block logs
`coolant_temp`, `intake_temp`, `rpm`, `speed` and `throttle_pos`, and nothing
oil-related appears in any of the four sessions. So the starting plot below is
the two temperatures that *are* logged. If you want oil temp on future sessions
it is OBD-II PID `0x5C` (engine oil temperature) — add it as a channel in
RaceChrono, assuming your adapter and ECU expose it. Many do not; a lot of cars
only report it over a manufacturer-specific PID, if at all.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Find the project root (the folder holding `racechrono/`), wherever this
# notebook is opened from.
ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "racechrono").is_dir():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from racechrono import csv_io, geo, laps as laps_mod

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

IMPERIAL = True                       # mph / degF. Set False for m/s and degC.
MPS_TO_MPH = 2.2369362920544


def sessions(data_dir="Raw Data"):
    return sorted((ROOT / data_dir).glob("*.csv"))


def load(path):
    """Full CSV v3 load: units validated, duplicate channel names resolved."""
    return csv_io.read_run(path)


def channels(run):
    """What this session actually logged, with units and ranges."""
    rows = []
    for c in run.df.columns:
        v = run.df[c]
        n = int(v.notna().sum())
        rows.append({"channel": c,
                     "unit": run.units.get(c) or "",
                     "device": run.devices.get(c) or "",
                     "n": n,
                     "min": round(float(v.min()), 3) if n else None,
                     "max": round(float(v.max()), 3) if n else None,
                     "constant": bool(v.nunique(dropna=True) <= 1)})
    return pd.DataFrame(rows).set_index("channel")


def convert(values, unit):
    """SI in the file, familiar units on the axis."""
    if IMPERIAL:
        if unit == "m/s":
            return values * MPS_TO_MPH, "mph"
        if unit == ".C":
            return values * 9.0 / 5.0 + 32.0, "\u00b0F"
    return values, (unit or "")


def _axis(run, x):
    if x == "time":
        return run.values("elapsed_time"), "session time (s)"
    if x == "distance":
        return run.values("distance_traveled") / 1000.0, "distance (km)"
    raise ValueError("x must be 'time' or 'distance'")

## Plot helpers

`plot` stacks a subplot per unit, so mixing rpm with throttle percent does not
squash one of them flat. Thin vertical lines mark start/finish crossings.

In [ ]:
def plot(run, names, x="time", mark_laps=True, good_only=False, width=13):
    """Channels against time or distance, one subplot per unit."""
    names = [names] if isinstance(names, str) else list(names)
    cols = [run.col(n) for n in names]          # resolves aliases e.g. rpm_obd
    xv, xlabel = _axis(run, x)
    keep = run.good_mask() if good_only else np.ones(len(run.df), bool)

    groups = {}
    for name, col in zip(names, cols):
        groups.setdefault(run.units.get(col) or "", []).append((name, col))

    fig, axes = plt.subplots(len(groups), 1, sharex=True, squeeze=False,
                             figsize=(width, max(3.0, 2.6 * len(groups))))
    axes = axes[:, 0]
    for ax, (unit, members) in zip(axes, groups.items()):
        out_unit = unit
        for name, col in members:
            y = run.df[col].to_numpy(float).copy()
            y[~keep] = np.nan
            y, out_unit = convert(y, unit)
            ax.plot(xv, y, lw=0.8, label=name)
        ax.set_ylabel(out_unit or "value")
        ax.legend(loc="upper right", fontsize=8, ncols=2)

    if mark_laps:
        for i in laps_mod.crossing_indices(run.values("lap_number")):
            for ax in axes:
                ax.axvline(xv[i], color="k", lw=0.5, alpha=0.25)

    axes[-1].set_xlabel(xlabel)
    axes[0].set_title("%s  --  %s" % (run.stem, ", ".join(names)), fontsize=10)
    fig.tight_layout()
    return fig


def plot_laps(run, name, laps=None, best_n=None, width=13):
    """One channel, one line per lap, against distance into the lap."""
    col = run.col(name)
    unit = run.units.get(col) or ""
    got = laps_mod.laps(run)
    if laps:
        got = [l for l in got if l.number in laps]
    if best_n:
        got = sorted(got, key=lambda l: l.duration)[:best_n]
    dist = run.values("distance_traveled")

    fig, ax = plt.subplots(figsize=(width, 5))
    for lp in sorted(got, key=lambda l: l.number):
        s = slice(lp.i0, lp.i1)
        y, out_unit = convert(run.df[col].to_numpy(float)[s], unit)
        ax.plot(dist[s] - dist[lp.i0], y, lw=0.9,
                label="lap %d  %.1fs" % (lp.number, lp.duration))
    ax.set_xlabel("distance into lap (m)")
    ax.set_ylabel(out_unit or "value")
    ax.legend(fontsize=8, ncols=3)
    ax.set_title("%s  --  %s by lap" % (run.stem, name), fontsize=10)
    fig.tight_layout()
    return fig


def plot_track(run, name, lap=None, cmap="viridis", size=4, figsize=(7, 8)):
    """The driven line, coloured by a channel. Positions failing the accuracy
    gate are dropped rather than drawn in the wrong place."""
    col = run.col(name)
    unit = run.units.get(col) or ""
    if lap is not None:
        lp = [l for l in laps_mod.laps(run) if l.number == lap][0]
        sl = slice(lp.i0, lp.i1)
        title = "%s -- %s, lap %d (%.1fs)" % (run.stem, name, lap, lp.duration)
    else:
        sl = slice(None)
        title = "%s -- %s, whole session" % (run.stem, name)

    good = run.good_mask()[sl]
    lat = run.values("latitude")[sl][good]
    lon = run.values("longitude")[sl][good]
    y, out_unit = convert(run.df[col].to_numpy(float)[sl][good], unit)
    xm, ym = geo.to_xy(lat, lon, *run.mean_latlon())

    fig, ax = plt.subplots(figsize=figsize)
    sc = ax.scatter(xm, ym, c=y, s=size, cmap=cmap, linewidths=0)
    ax.set_aspect("equal")
    ax.set_xlabel("east (m)")
    ax.set_ylabel("north (m)")
    ax.set_title(title, fontsize=10)
    fig.colorbar(sc, ax=ax, label="%s (%s)" % (name, out_unit or ""))
    fig.tight_layout()
    return fig

## Pick a session

In [ ]:
paths = sessions()
for i, p in enumerate(paths):
    print(i, p.name)

run = load(paths[1])          # <-- change the index to switch session
print("\n%s: %.1f s, %d rows, %d complete laps"
      % (run.stem, run.duration, len(run.df), len(laps_mod.laps(run))))

## What is in it

`constant = True` means the channel never changes, so there is nothing to plot —
`altitude` and `fix_type` are both dead in this dataset.

In [ ]:
channels(run)

## Starting plot: temperatures

Coolant and intake, the only temperature channels present. Intake drops sharply
once the car is moving and airflow starts; coolant climbs and then plateaus.

In [ ]:
temps = [c for c in run.df.columns if run.units.get(c) == ".C"]
print("temperature channels:", temps)
plot(run, temps);

## Pick your own channels

Edit the list and re-run. Names come from the table above; aliases like
`rpm_obd` or `speed_gps` also resolve.

In [ ]:
CHANNELS = ["speed_gps", "rpm", "throttle_pos"]

plot(run, CHANNELS);
# plot(run, CHANNELS, x="distance")         # against distance instead of time
# plot(run, ["lateral_acc", "longitudinal_acc", "combined_acc"]);
# plot(run, ["accuracy"], good_only=False)  # where the GPS degraded

## Compare laps

In [ ]:
plot_laps(run, "speed_gps", best_n=3);
# plot_laps(run, "throttle_pos", laps=[4, 5])

## Colour the driven line by a channel

In [ ]:
plot_track(run, "speed_gps", lap=6);
# plot_track(run, "combined_acc", lap=6, cmap="magma")
# plot_track(run, "accuracy")               # where fixes went bad

---

### Notes

- Values are SI in the file. `IMPERIAL = True` at the top converts m/s to mph
  and `.C` to degF on the axis only; the underlying data is untouched.
- `run.values("name")` gives a raw numpy array, `run.df` the whole frame, if you
  want to do something the helpers do not cover.
- `run.good_mask()` is the accuracy gate: `accuracy <= 10 m`. That channel floors
  at exactly 10.0, so the gate is `<=`, not `<`. `plot_track` applies it always;
  `plot` only when you pass `good_only=True`.
- `lap_number` is NaN until the first start/finish crossing, so the out-lap has
  no lap number. `laps_mod.laps(run)` returns only complete laps.
- `bearing` is interpolated up from about 1 Hz, so differentiating it between
  adjacent 20 Hz samples gives noise, not yaw rate. Difference over 0.5 s or more.
- `lean_angle` is a derived motorcycle channel and correlates -0.97 with
  `lateral_acc`. It means nothing on a car.